In [ ]:
%run setup.py

# 06 · Comparación e interpretación de los tres enfoques

Este notebook reúne los resultados de los tres modelos y los interpreta dentro
del escenario de precios de viviendas. Los tres enfoques responden preguntas
distintas, por lo que la comparación es conceptual y de resultados, no de una
misma métrica.

In [ ]:
# CELDA 6: Resultados del modelo supervisado
df = load_processed()

columnas_region = [c for c in df.columns if c.startswith("Regionname_")]
features = ["Rooms", "BuildingArea", "Antiguedad"] + columnas_region
X = StandardScaler().fit_transform(df[features])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modelo = LinearRegression().fit(X_train, y_train)
y_pred = modelo.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Regresión Lineal -> R²={r2:.4f} | MAE=${mae:,.0f} | RMSE=${rmse:,.0f}")

In [ ]:
# Resultados del clustering
features_cluster = ["Rooms", "BuildingArea", "Distance", "Antiguedad"]
dfc = df.dropna(subset=features_cluster).copy()
Xc = StandardScaler().fit_transform(dfc[features_cluster])

km = KMeans(n_clusters=3, random_state=42, n_init=10)
dfc["Segmento"] = km.fit_predict(Xc)
sil = silhouette_score(Xc, dfc["Segmento"], sample_size=5000, random_state=42)
print(f"K-Means (k=3) -> silhouette={sil:.4f}")
display(dfc.groupby("Segmento")[["Price", "Rooms", "BuildingArea"]].mean().round(2))

In [ ]:
# Resultados del agente de refuerzo (misma simulación del notebook 05)
random.seed(42)
precios = {"Alto": 1_000_000, "Medio": 800_000, "Bajo": 500_000}
probabilidades_venta = {"Alto": 0.10, "Medio": 0.60, "Bajo": 0.85}
acciones = list(precios)
epsilon, episodios = 0.1, 1000
intentos = {a: 0 for a in acciones}
recompensas = {a: 0 for a in acciones}
historial = []

for _ in range(episodios):
    if random.random() < epsilon:
        accion = random.choice(acciones)
    else:
        promedios = {a: (recompensas[a] / intentos[a] if intentos[a] else 0) for a in acciones}
        accion = max(promedios, key=promedios.get)
    intentos[accion] += 1
    if random.random() < probabilidades_venta[accion]:
        recompensas[accion] += precios[accion]
    historial.append(sum(recompensas.values()))

mejor = max(acciones, key=lambda a: recompensas[a] / intentos[a] if intentos[a] else 0)
print(f"Multi-Armed Bandits -> mejor estrategia: {mejor}")
print(f"Recompensa total acumulada: ${historial[-1]:,.0f}")

## 6.1 Tabla comparativa

In [ ]:
comparacion = pd.DataFrame([
    {
        "Enfoque": "Supervisado",
        "Algoritmo": "Regresión Lineal",
        "Pregunta": "¿Cuánto vale una vivienda?",
        "Resultado principal": f"R² = {r2:.3f}; MAE = ${mae:,.0f}",
        "Interpretación": "Explica ~40% de la varianza; útil como referencia, pero omite factores.",
    },
    {
        "Enfoque": "No supervisado",
        "Algoritmo": "K-Means (k=3)",
        "Pregunta": "¿Qué perfiles de vivienda existen?",
        "Resultado principal": f"Silhouette = {sil:.3f}",
        "Interpretación": "Separa viviendas económicas, medianas y premium.",
    },
    {
        "Enfoque": "Refuerzo",
        "Algoritmo": "Multi-Armed Bandits (ε-greedy)",
        "Pregunta": "¿Qué precio de publicación conviene?",
        "Resultado principal": f"Mejor acción = {mejor}",
        "Interpretación": "Aprende a priorizar la estrategia con mayor ingreso esperado.",
    },
])
display(comparacion)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(["MAE", "RMSE"], [mae, rmse], color=["steelblue", "indianred"])
axes[0].set_title("Error del modelo supervisado (AUD)")
axes[0].set_ylabel("AUD")

dfc["Segmento"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="teal")
axes[1].set_title("Tamaño de los segmentos K-Means")
axes[1].set_xlabel("Segmento")
axes[1].set_ylabel("Viviendas")

axes[2].plot(historial)
axes[2].set_title("Recompensa acumulada del agente")
axes[2].set_xlabel("Episodio")
axes[2].set_ylabel("AUD")

plt.tight_layout()
plt.show()

## 6.2 Conclusiones

- **Supervisado:** la regresión lineal permite estimar el precio e interpretar
  coeficientes, pero su R² moderado muestra que el precio depende de factores
  no incluidos (estado de la vivienda, calidad, submercado).
- **No supervisado:** K-Means con `k=3` entrega segmentos interpretables que
  pueden apoyar estrategias comerciales diferenciadas.
- **Refuerzo:** el agente aprende qué estrategia de precio maximiza ingresos,
  equilibrando probabilidad de venta y margen.

**Limitaciones generales:** alta proporción de nulos en `BuildingArea` y
`YearBuilt`, outliers en el precio y un entorno de refuerzo simulado.
**Mejoras:** imputación más fina, modelos no lineales, variables adicionales y
un bandit contextual que use características de cada vivienda.